In [ ]:
from pathlib import Path
from PIL import Image
from collections import Counter

image_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

dataset = Path("archive/images")

sizes = []
formats = Counter()

for img_path in dataset.rglob("*"):
    if img_path.suffix.lower() not in image_ext:
        continue

    try:
        img = Image.open(img_path)

        sizes.append(img.size)

        formats[img.format] += 1

    except Exception:
        print("Corrupted:", img_path)

print("Total images:", len(sizes))
print()

print("Formats")
print(formats)

print()

widths = [w for w, h in sizes]
heights = [h for w, h in sizes]

print("Width")
print(min(widths), max(widths))

print("Height")
print(min(heights), max(heights))

Total images: 0

Formats
Counter()

Width


ValueError: min() iterable argument is empty

3. Exploratory Data Analysis (EDA)

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image
import hashlib

# ============================================================
# PATH
# ============================================================

ROOT = Path(__file__).parent if '__file__' in locals() else Path(".")

IMAGE_DIR = ROOT / "archive" / "images"
LABEL_DIR = ROOT / "archive" / "labels"

CLASS_FILE = ROOT / "archive" / "classes.txt"

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
 
# ============================================================
# LOAD IMAGES & LABELS
# ============================================================

image_files = {
    p.stem: p
    for p in IMAGE_DIR.iterdir()
    if p.suffix.lower() in IMAGE_EXT
}

label_files = {
    p.stem: p
    for p in LABEL_DIR.glob("*.txt")
}

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print(f"Images : {len(image_files)}")
print(f"Labels : {len(label_files)}")

DATASET SUMMARY
Images : 3216
Labels : 3216


3.1 Images & Labels Mismatch

In [ ]:
# ============================================================
# IMAGE - LABEL CONSISTENCY
# ============================================================

missing_labels = sorted(set(image_files) - set(label_files))
orphan_labels = sorted(set(label_files) - set(image_files))

print("\nIMAGE / LABEL CONSISTENCY")
print("-" * 60)

print(f"Images without labels : {len(missing_labels)}")
print(f"Orphan labels         : {len(orphan_labels)}")

if missing_labels:
    print("\nFirst 10 missing labels")
    for x in missing_labels[:10]:
        print(x)

if orphan_labels:
    print("\nFirst 10 orphan labels")
    for x in orphan_labels[:10]:
        print(x)


IMAGE / LABEL CONSISTENCY
------------------------------------------------------------
Images without labels : 0
Orphan labels         : 0


3.2 Class Distribution

In [ ]:
# ============================================================
# LOAD CLASSES
# ============================================================

classes = []

with open(CLASS_FILE, encoding="utf-8") as f:
    classes = [line.strip() for line in f if line.strip()]

print("\nNumber of classes:", len(classes))

# ============================================================
# CLASS DISTRIBUTION
# ============================================================

class_counter = Counter()

invalid_class = []

empty_labels = []

for label in label_files.values():

    lines = label.read_text().strip().splitlines()

    if len(lines) == 0:
        empty_labels.append(label.name)
        continue

    for line in lines:

        parts = line.split()

        if len(parts) < 5:
            continue

        try:
            cls = int(parts[0])

            if cls >= len(classes):
                invalid_class.append(label.name)

            else:
                class_counter[cls] += 1

        except:
            continue

print("\nCLASS DISTRIBUTION")
print("-" * 60)

total_objects = sum(class_counter.values())

print(f"Total objects : {total_objects}")

for idx, name in enumerate(classes):

    print(
        f"{idx:>2}  "
        f"{name:<25}"
        f"{class_counter[idx]:>8}"
    )

print()

print("Empty label files :", len(empty_labels))
print("Invalid class ids :", len(set(invalid_class)))


Number of classes: 52

CLASS DISTRIBUTION
------------------------------------------------------------
Total objects : 8334
 0  W.224                         312
 1  W.205c                         22
 2  P.102                         470
 3  R.302a                        454
 4  W.205a                         33
 5  W.207                         256
 6  W.201a                        182
 7  P.123a                        247
 8  I.434a                        213
 9  R.303                          43
10  P.130                        1071
11  I.409                          26
12  R.415a                        280
13  W.245a                        372
14  P.106a*Xe tải                  85
15  W.203c                         29
16  P.117*                         73
17  P.124a*                       105
18  P.107                         214
19  P.124d                         29
20  P.103a                         52
21  W.203b                         11
22  W.221b                        131
2

3.3 Image Quality

In [ ]:
# ============================================================
# IMAGE TYPE CHECK
# ============================================================

gray_images = []
rgba_images = []
small_images = []

for path in image_files.values():

    with Image.open(path) as img:

        w, h = img.size

        if img.mode == "L":
            gray_images.append(path.name)

        elif img.mode == "RGBA":
            rgba_images.append(path.name)

        if w < 100 or h < 100:
            small_images.append(path.name)

print("\nIMAGE QUALITY")
print("-" * 60)

print("Grayscale :", len(gray_images))
print("RGBA      :", len(rgba_images))
print("Small     :", len(small_images))


IMAGE QUALITY
------------------------------------------------------------
Grayscale : 0
RGBA      : 0
Small     : 0


3.4 Duplicate Detection

In [ ]:
# ============================================================
# DUPLICATE FILES
# ============================================================

hash_table = {}

duplicate_files = []

for path in image_files.values():

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            sha.update(chunk)

    digest = sha.hexdigest()

    if digest in hash_table:

        duplicate_files.append(
            (hash_table[digest].name, path.name)
        )

    else:

        hash_table[digest] = path

print("\nDUPLICATE FILES")
print("-" * 60)

print("Duplicate images :", len(duplicate_files))

for a, b in duplicate_files[:10]:
    print(a, "<->", b)


DUPLICATE FILES
------------------------------------------------------------
Duplicate images : 0


3.5 Class Imbalance Report


In [ ]:
# ============================================================
# CLASS IMBALANCE
# ============================================================

print("\nCLASS IMBALANCE")
print("-" * 60)

if total_objects > 0:

    max_cls = max(class_counter.values())
    min_cls = min(v for v in class_counter.values() if v > 0)

    print("Largest class :", max_cls)
    print("Smallest class:", min_cls)
    print(f"Imbalance ratio : {max_cls/min_cls:.2f}")

    print()

    for idx, name in enumerate(classes):

        count = class_counter[idx]

        percent = count / total_objects * 100

        print(f"{idx:>2} {name:<25}{count:>7} ({percent:6.2f}%)")


CLASS IMBALANCE
------------------------------------------------------------
Largest class : 1071
Smallest class: 3
Imbalance ratio : 357.00

 0 W.224                        312 (  3.74%)
 1 W.205c                        22 (  0.26%)
 2 P.102                        470 (  5.64%)
 3 R.302a                       454 (  5.45%)
 4 W.205a                        33 (  0.40%)
 5 W.207                        256 (  3.07%)
 6 W.201a                       182 (  2.18%)
 7 P.123a                       247 (  2.96%)
 8 I.434a                       213 (  2.56%)
 9 R.303                         43 (  0.52%)
10 P.130                       1071 ( 12.85%)
11 I.409                         26 (  0.31%)
12 R.415a                       280 (  3.36%)
13 W.245a                       372 (  4.46%)
14 P.106a*Xe tải                 85 (  1.02%)
15 W.203c                        29 (  0.35%)
16 P.117*                        73 (  0.88%)
17 P.124a*                      105 (  1.26%)
18 P.107                     